# 05 — SHAP Reason Codes

Explain per-transaction predictions using **SHAP TreeExplainer**. This produces:
1. Top-5 adverse-action reason codes per transaction (required for regulatory compliance)
2. A review queue of the 5 highest expected-loss cases (amount × P(fraud))

Output: appended to `data_export.json`

In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import shap

DATA_DIR    = Path("../../..") / "data" / "ieee-fraud-detection"
DEMO_DIR    = Path("..") / "demo"
EXPORT_PATH = DEMO_DIR / "data_export.json"
MODEL_PATH  = Path("../../..") / "models" / "fraud_xgb_calibrated.pkl"

N_EXPLAIN = 1000   # 500 fraud + 500 legit
TOP_K     = 5


/Users/stuartchen/Documents/Claude/Projects/AI workflow demo/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load model

In [2]:
with open(MODEL_PATH, "rb") as f:
    bundle = pickle.load(f)

xgb_base      = bundle["xgb_base"]
iso           = bundle["iso"]
feature_names = bundle["feature_names"]
print(f"Model loaded — {len(feature_names)} features")


Model loaded — 410 features


## 2. Reconstruct test feature matrix

In [3]:
df_pred  = pd.read_parquet(DATA_DIR / "test_predictions.parquet")
eng      = pd.read_parquet(DATA_DIR / "train_engineered.parquet")
eng_test = (
    eng[eng["TransactionID"].isin(df_pred["TransactionID"])]
    .set_index("TransactionID")
    .loc[df_pred["TransactionID"].values]
    .reset_index()
)

drop_cols = ["TransactionID","TransactionDT","isFraud",
             "card2","card3","card5","addr1","addr2",
             "P_emaildomain","R_emaildomain","DeviceInfo","DeviceType",
             "card4","card6","ProductCD",
             "M1","M2","M3","M4","M5","M6","M7","M8","M9",
             "id_12","id_15","id_16","id_23","id_27","id_28",
             "id_29","id_30","id_31","id_32","id_33","id_34",
             "id_35","id_36","id_37","id_38"]
drop_cols = [c for c in drop_cols if c in eng_test.columns]

X_test = eng_test.drop(columns=drop_cols)
for col in X_test.select_dtypes("object").columns:
    X_test[col] = X_test[col].astype("category").cat.codes
for col in feature_names:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[feature_names]
print(f"Test features: {X_test.shape}")


Test features: (118108, 410)


## 3. Stratified sample — 500 fraud + 500 legit

In [4]:
fraud_idx  = eng_test[eng_test["isFraud"] == 1].index[:N_EXPLAIN // 2]
legit_idx  = eng_test[eng_test["isFraud"] == 0].index[:N_EXPLAIN // 2]
sample_idx = fraud_idx.tolist() + legit_idx.tolist()

pos_map    = {orig: pos for pos, orig in enumerate(eng_test.index)}
sample_pos = [pos_map[i] for i in sample_idx if i in pos_map]

X_sample       = X_test.iloc[sample_pos]
y_sample       = eng_test["isFraud"].iloc[sample_pos].values
tid_sample     = eng_test["TransactionID"].iloc[sample_pos].values
y_prob_sample  = df_pred.set_index("TransactionID").loc[tid_sample, "y_prob"].values

print(f"Sample: {len(X_sample)} transactions  |  fraud: {y_sample.mean():.0%}")


Sample: 1000 transactions  |  fraud: 50%


## 4. Compute SHAP values

`TreeExplainer` is exact and fast for tree-based models — no approximation needed.

In [5]:
explainer   = shap.TreeExplainer(xgb_base)
shap_values = explainer.shap_values(X_sample)

print(f"SHAP matrix shape: {shap_values.shape}")
print(f"Columns: {shap_values.shape[1]}  (one per feature)")


SHAP matrix shape: (1000, 410)
Columns: 410  (one per feature)


## 5. Build per-transaction reason code records

In [6]:
UI_FEATURE_KEYS = [
    "TransactionAmt", "addr_age_days", "email_domain_mismatch",
    "new_acct_highval_express", "velocity_count_1h", "velocity_value_1h",
    "velocity_count_24h", "velocity_value_24h", "device_linkage_count",
    "has_identity", "dist1", "c_total",
]

records = []
for i, (tid, y_true, y_prob) in enumerate(zip(tid_sample, y_sample, y_prob_sample)):
    sv      = shap_values[i]
    top_idx = np.argsort(np.abs(sv))[::-1][:TOP_K]
    top5    = [
        {"feature": feature_names[j], "value": round(float(X_sample.iloc[i, j]), 4),
         "shap": round(float(sv[j]), 4)}
        for j in top_idx
    ]
    key_features = {
        k: round(float(X_sample.iloc[i][k]), 4)
        for k in UI_FEATURE_KEYS if k in X_sample.columns
    }
    records.append({
        "transaction_id": int(tid),
        "y_true": int(y_true),
        "fraud_prob": round(float(y_prob), 4),
        "shap_top5": top5,
        "key_features": key_features,
    })

print(f"Built {len(records)} SHAP records")
print(f"\nSample — transaction {records[0]['transaction_id']}:")
for r in records[0]["shap_top5"]:
    print(f"  {r['feature']:30s}  shap={r['shap']:+.4f}  value={r['value']}")


Built 1000 SHAP records

Sample — transaction 3059645:
  C1                              shap=+1.6301  value=14.0
  V258                            shap=+0.9097  value=2.0
  C6                              shap=+0.4204  value=10.0
  V201                            shap=+0.4079  value=2.0
  card1                           shap=-0.3237  value=16746.0


## 6. Build review queue

Sort by **expected loss = amount × P(fraud)** — prioritises high-value high-confidence cases for analyst review.

In [7]:
queue_candidates = [r for r in records if r["fraud_prob"] > 0.3]
queue_candidates.sort(
    key=lambda r: r["key_features"].get("TransactionAmt", 0) * r["fraud_prob"],
    reverse=True,
)
review_queue = queue_candidates[:5]

print(f"Queue candidates (P>0.3): {len(queue_candidates)}")
print(f"\nTop 5 review cases:")
for i, r in enumerate(review_queue):
    amt = r["key_features"].get("TransactionAmt", 0)
    print(f"  {i+1}. TxID={r['transaction_id']}  amount=${amt:.0f}  P(fraud)={r['fraud_prob']:.3f}  expected_loss=${amt*r['fraud_prob']:.0f}")


Queue candidates (P>0.3): 299

Top 5 review cases:
  1. TxID=3192215  amount=$1000  P(fraud)=1.000  expected_loss=$1000
  2. TxID=3227454  amount=$884  P(fraud)=0.878  expected_loss=$776
  3. TxID=3374602  amount=$1000  P(fraud)=0.754  expected_loss=$754
  4. TxID=3469805  amount=$1436  P(fraud)=0.509  expected_loss=$731
  5. TxID=3393433  amount=$1104  P(fraud)=0.571  expected_loss=$631


## 7. Export to data_export.json

In [8]:
with open(EXPORT_PATH) as f:
    export = json.load(f)

export["transactions"]  = records
export["review_queue"]  = review_queue

with open(EXPORT_PATH, "w") as f:
    json.dump(export, f)

print(f"Saved → {EXPORT_PATH}")
print(f"Total keys in export: {list(export.keys())}")


Saved → ../demo/data_export.json
Total keys in export: ['pr_curve', 'pr_auc', 'optimal_threshold', 'optimal_precision', 'optimal_recall', 'c_fn', 'c_fp', 'capacity_per_day', 'adversarial', 'transactions', 'review_queue', 'cost_optimal_threshold']
